In [29]:
import os
import pandas as pd
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

# 儲存資料名稱、訓練資料、訓練標籤與測試資料
dataset_names = []
X_trains = []
y_trains = []
X_tests = []

# 讀取資料
for folder_name in os.listdir("./Competition_data"):
    dataset_names.append(folder_name)
    X_trains.append(pd.read_csv(f"./Competition_data/{folder_name}/X_train.csv", header=0))
    y_trains.append(pd.read_csv(f"./Competition_data/{folder_name}/y_train.csv", header=0))
    X_tests.append(pd.read_csv(f"./Competition_data/{folder_name}/X_test.csv", header=0))

# 設置 K-fold 交叉驗證
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 設置模型的超參數搜索範圍，增加正則化參數
param_grid = {
    'xgb': {
        'learning_rate': uniform(0.01, 0.1),
        'max_depth': randint(3, 6),  # 降低模型複雜度
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3),
        'n_estimators': randint(100, 200),
        'reg_alpha': uniform(0, 1),  # L1 正則化
        'reg_lambda': uniform(0, 1),  # L2 正則化
    },
    'cat': {
        'learning_rate': uniform(0.01, 0.1),
        'depth': randint(3, 6),  # 降低模型複雜度
        'iterations': randint(100, 200),
        'l2_leaf_reg': uniform(1, 5)  # L2 正則化
    },
    'lgb': {
        'learning_rate': uniform(0.01, 0.1),
        'max_depth': randint(3, 6),  # 降低模型複雜度
        'num_leaves': randint(20, 50),
        'n_estimators': randint(100, 200),
        'min_data_in_leaf': randint(20, 50),  # 控制每個葉子內的最小數據數
        'reg_alpha': uniform(0, 1),
        'reg_lambda': uniform(0, 1)
    }
}

# 建立模型並進行訓練和預測
for i in range(len(dataset_names)):
    print(f"Processing dataset: {dataset_names[i]}")

    # 取得資料集
    X_train, y_train = X_trains[i], y_trains[i].squeeze()

    # 定義模型
    xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc', random_state=42)
    cat_model = cb.CatBoostClassifier(loss_function='Logloss', eval_metric='AUC', random_state=42, verbose=0)
    lgb_model = lgb.LGBMClassifier(objective='binary', random_state=42)
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)  # 控制模型複雜度
    gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)  # 控制模型複雜度
    et_model = ExtraTreesClassifier(n_estimators=100, max_depth=4, random_state=42)  # 控制模型複雜度

    # 使用 RandomizedSearchCV 來尋找最佳參數
    random_search_xgb = RandomizedSearchCV(xgb_model, param_grid['xgb'], n_iter=25, scoring='roc_auc', cv=kf, random_state=42, n_jobs=-1)
    random_search_cat = RandomizedSearchCV(cat_model, param_grid['cat'], n_iter=25, scoring='roc_auc', cv=kf, random_state=42, n_jobs=-1)
    random_search_lgb = RandomizedSearchCV(lgb_model, param_grid['lgb'], n_iter=25, scoring='roc_auc', cv=kf, random_state=42, n_jobs=-1)

    # 訓練隨機搜尋模型
    random_search_xgb.fit(X_train, y_train)
    random_search_cat.fit(X_train, y_train)
    random_search_lgb.fit(X_train, y_train)

    # 最佳參數的模型
    best_xgb_model = random_search_xgb.best_estimator_
    best_cat_model = random_search_cat.best_estimator_
    best_lgb_model = random_search_lgb.best_estimator_
    
    print(f"Best XGBoost AUC for {dataset_names[i]}: {random_search_xgb.best_score_:.4f}")
    print(f"Best CatBoost AUC for {dataset_names[i]}: {random_search_cat.best_score_:.4f}")
    print(f"Best LightGBM AUC for {dataset_names[i]}: {random_search_lgb.best_score_:.4f}")

    # 建立加權投票模型
    voting_model = VotingClassifier(
        estimators=[
            ('xgb', best_xgb_model),
            ('cat', best_cat_model),
            ('lgb', best_lgb_model),
            ('rf', rf_model),
            ('gb', gb_model),
            ('et', et_model)
        ],
        voting='soft',
        weights=[2, 2, 1.5, 1, 1, 1]
    )

    # 訓練投票模型
    voting_model.fit(X_train, y_train)

    # 計算交叉驗證 AUC
    y_val_pred = voting_model.predict_proba(X_train)[:, 1]
    auc = roc_auc_score(y_train, y_val_pred)
    print(f"Validation AUC for {dataset_names[i]}: {auc:.4f}")

    # 使用投票模型對測試資料進行預測
    y_predict_proba = voting_model.predict_proba(X_tests[i])[:, 1]
    
    # 儲存預測結果至 CSV
    proba_df = pd.DataFrame(y_predict_proba, columns=['y_predict_proba'])
    proba_df.to_csv(f'./Competition_data/{dataset_names[i]}/y_predict.csv', index=False)
    print(f"Saved predictions to ./Competition_data/{dataset_names[i]}/y_predict.csv\n")

print("All datasets processed.")

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Training AUC: 0.8258
Meta-Model 預測結果已儲存至 'meta_model_y_predict.csv'
